# Transfer-Learning / Fine-Tuning Showcase for YOLO

## 1. Setup and Imports

 This section sets up paths and imports necessary libraries.

In [ ]:
import os
from ultralytics import YOLO

# Define paths for reproducibility
BASE_MODEL_PATH = "./yolo11n.pt"
BASE_IMG_DIR = "./base/score_base_in"
BASE_OUTPUT_DIR = "./base/score_base_out"

# Absolute path: the YAML lists its image folders relative to its own location.
CUSTOM_YAML_PATH = os.path.abspath("./custom/showcase_dataset.yaml")
CUSTOM_PROJECT_DIR = "./custom"
CUSTOM_TRAIN_NAME = "fine-tune"
CUSTOM_TEST_IMG_DIR = "./custom/images/test"
CUSTOM_SCORE_DIR = "./custom/score_custom_out"

# Training parameters
EPOCHS = 50
BATCH = 8
IMG_SIZE = 640
# n_train_imgs = 38
# n_val_imgs = 12
# model = "yolo11n.pt"

**EPOCHS**: The number of times the entire training dataset is passed through the model during training. More epochs can help the model learn better, but too many can lead to overfitting.
**BATCH** (batch size): The number of images the model processes at once before updating its internal parameters. A larger batch size can speed up training (if you have enough GPU memory) but may require more resources.
**IMG_SIZE** (image size): The target size (usually width and height in pixels) to which all training and inference images are resized. Larger sizes can improve detection accuracy but use more memory and computation.

## 2. Scoring Images with a Pretrained YOLO Model

Use a YOLO model pretrained on the COCO dataset (80 classes) to detect objects in your images.
Images and detection results will be saved for inspection.

In [2]:
# Load pretrained YOLO model
base_model = YOLO(BASE_MODEL_PATH)

# List available test images
test_images = os.listdir(BASE_IMG_DIR)

# Inference on each image. Results, labels, and confidence values are saved.
for img_name in test_images:
    img_path = os.path.join(BASE_IMG_DIR, img_name)
    results = base_model(
        img_path,
        save=True,          # Save resulting images with drawn detections
        save_txt=True,      # Save labels in YOLO format
        save_conf=True,     # Save confidences in labels
        project=BASE_OUTPUT_DIR,
        exist_ok=True
    )

print(f"Detection results are saved in '{BASE_OUTPUT_DIR}'")


image 1/1 /mnt/batch/tasks/shared/LS_root/mounts/clusters/at-beve-dev-michael/code/Users/Michael.Bach/img-reco/showcase/base/score_base_in/cuttlery.jpg: 448x640 4 forks, 2 knifes, 3 spoons, 113.2ms
Speed: 3.6ms preprocess, 113.2ms inference, 1.6ms postprocess per image at shape (1, 3, 448, 640)
Results saved to showcase/base/score_base_out/predict
5 labels saved to showcase/base/score_base_out/predict/labels

image 1/1 /mnt/batch/tasks/shared/LS_root/mounts/clusters/at-beve-dev-michael/code/Users/Michael.Bach/img-reco/showcase/base/score_base_in/dinner_table.jpg: 448x640 2 wine glasss, 1 cup, 3 forks, 2 knifes, 1 spoon, 3 chairs, 5 potted plants, 1 dining table, 2 vases, 77.9ms
Speed: 2.7ms preprocess, 77.9ms inference, 1.2ms postprocess per image at shape (1, 3, 448, 640)
Results saved to showcase/base/score_base_out/predict
5 labels saved to showcase/base/score_base_out/predict/labels

image 1/1 /mnt/batch/tasks/shared/LS_root/mounts/clusters/at-beve-dev-michael/code/Users/Michael.B

## 3. Fine-Tuning the Pretrained Model on Custom Data

Next, we fine-tune the pretrained YOLO model using your custom dataset.
The YAML file should reference your train/val dataset locations and class names.

In [8]:
# Start fine-tuning (transfer learning)
custom_model = YOLO(BASE_MODEL_PATH)

train_results = custom_model.train(
    data=CUSTOM_YAML_PATH,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    project=CUSTOM_PROJECT_DIR,
    name=CUSTOM_TRAIN_NAME
)

print(f"Fine-tuned model training is complete. Check training logs and weights at '{CUSTOM_PROJECT_DIR}/{CUSTOM_TRAIN_NAME}'")

New https://pypi.org/project/ultralytics/8.3.192 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.161 🚀 Python-3.10.18 torch-2.7.1+cu126 CPU (Intel Xeon Platinum 8272CL 2.60GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./showcase/custom/showcase_dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=./yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=fine-tune2, nbs=64, nms=Fal

train: Scanning /mnt/batch/tasks/shared/LS_root/mounts/clusters/at-beve-dev-michael/code/Users/Michael.Bach/img-reco/showcase/custom/labels/train.cache... 38 images, 0 backgrounds, 0 corrupt: 100%|██████████| 38/38 [00:00<?, ?it/s]
/anaconda/envs/img-reco/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


val: Fast image access ✅ (ping: 0.1±0.0 ms, read: 6.5±1.9 MB/s, size: 268.2 KB)


val: Scanning /mnt/batch/tasks/shared/LS_root/mounts/clusters/at-beve-dev-michael/code/Users/Michael.Bach/img-reco/showcase/custom/labels/val.cache... 12 images, 0 backgrounds, 0 corrupt: 100%|██████████| 12/12 [00:00<?, ?it/s]

Plotting labels to showcase/custom/fine-tune2/labels.jpg... 



/anaconda/envs/img-reco/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


... (149 outputs left out)

                   all         12         12      0.934          1      0.995      0.645



50 epochs completed in 0.211 hours.
Optimizer stripped from showcase/custom/fine-tune2/weights/last.pt, 5.5MB
Optimizer stripped from showcase/custom/fine-tune2/weights/best.pt, 5.5MB

Validating showcase/custom/fine-tune2/weights/best.pt...
Ultralytics 8.3.161 🚀 Python-3.10.18 torch-2.7.1+cu126 CPU (Intel Xeon Platinum 8272CL 2.60GHz)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:01<00:00,  1.78s/it]


                   all         12         12       0.92          1      0.989      0.669
Speed: 1.2ms preprocess, 93.5ms inference, 0.0ms loss, 8.3ms postprocess per image
Results saved to showcase/custom/fine-tune2
Fine-tuned model training is complete. Check training logs and weights at './showcase/custom/fine-tune'


Description of the training output — the last block validates `best.pt`. YOLO keeps the round that scored best on the validation images (here epoch 39 of 50), not the last round, so these are the scores of the model used below:

- **Class:** Summary across all classes
- **Images:** 12 validation images were used
- **Instances:** 12 object instances detected
- **Box(P):** Precision (0.92) – proportion of correct detections among all detections
- **R:** Recall (1) – proportion of ground-truth objects that were correctly detected
- **mAP50:** Mean Average Precision at IoU 0.5 (0.989) – detects well if above 0.9
- **mAP50-95:** Mean average precision averaged over IoU thresholds 0.50 to 0.95 (0.669) – a stricter metric, usually lower

## 4. Using the Fine-Tuned Model for Inference

Load the resulting best model weights and run inference on your custom test set.

In [10]:
# Define path to best weights after fine-tuning
best_weights_path = os.path.join(CUSTOM_PROJECT_DIR, CUSTOM_TRAIN_NAME, "weights", "best.pt")

# Load the updated (fine-tuned) model
finetuned_model = YOLO(best_weights_path)

# List test images for custom dataset
custom_test_images = os.listdir(CUSTOM_TEST_IMG_DIR)

# Runs inference and saves outputs for each test image
for img_name in custom_test_images:
    img_path = os.path.join(CUSTOM_TEST_IMG_DIR, img_name)
    results = finetuned_model(
        img_path,
        save=True,
        save_txt=True,
        save_conf=True,
        project=CUSTOM_SCORE_DIR,
        exist_ok=True
    )

print(f"Inference with fine-tuned model done! Results saved in '{CUSTOM_SCORE_DIR}'")


image 1/1 /mnt/batch/tasks/shared/LS_root/mounts/clusters/at-beve-dev-michael/code/Users/Michael.Bach/img-reco/showcase/custom/images/test/743434fa-18.jpg: 448x640 1 spaghetti_bolo, 110.1ms
Speed: 3.1ms preprocess, 110.1ms inference, 1.2ms postprocess per image at shape (1, 3, 448, 640)
Results saved to showcase/custom/score_custom_out/predict
4 labels saved to showcase/custom/score_custom_out/predict/labels

image 1/1 /mnt/batch/tasks/shared/LS_root/mounts/clusters/at-beve-dev-michael/code/Users/Michael.Bach/img-reco/showcase/custom/images/test/ita_dishes.jpg: 512x640 6 spaghetti_bolos, 93.2ms
Speed: 2.9ms preprocess, 93.2ms inference, 1.2ms postprocess per image at shape (1, 3, 512, 640)
Results saved to showcase/custom/score_custom_out/predict
4 labels saved to showcase/custom/score_custom_out/predict/labels

image 1/1 /mnt/batch/tasks/shared/LS_root/mounts/clusters/at-beve-dev-michael/code/Users/Michael.Bach/img-reco/showcase/custom/images/test/pasta_dishes.jpg: 448x640 3 spaghett

## 5. Summary

- First, the script scores images with a COCO-pretrained YOLO model.
- Second, it fine-tunes that model with your custom labeled data.
- Finally, it uses the new fine-tuned weights to infer on new custom images.

Check the output directories for detection results and logs!

To share results without the training photos, run `uv run python export_results.py`: it applies both models to the openly licensed photos in `results/images/` and writes boxes, metrics and plots to `results/` (see the README).